In [ ]:
!pip install altair pandas openpyxl vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 47.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import altair as alt
df = pd.read_excel("CR y EN (por región).xlsm")

In [23]:
# Filtrar categorías CR, EN y EN-RARA
df_filtrado = df[df["CATEGORÍA VIGENTE"].isin(["CR", "EN", "EN-RARA"])]

# Contar especies por región y categoría
conteo = (
    df_filtrado
    .groupby(["REGIÓN", "CATEGORÍA VIGENTE"])
    .size()
    .reset_index(name="TOTAL")
)

# Totales generales
total_cr = conteo[conteo["CATEGORÍA VIGENTE"] == "CR"]["TOTAL"].sum()
total_en = conteo[conteo["CATEGORÍA VIGENTE"] == "EN"]["TOTAL"].sum()
total_en_rara = conteo[conteo["CATEGORÍA VIGENTE"] == "EN-RARA"]["TOTAL"].sum()

total_general = total_cr + total_en + total_en_rara

# Texto resumen superior
resumen = pd.DataFrame({
    "texto": [
        f"CR: {total_cr} especies   |   EN: {total_en} especies   |   EN-RARA: {total_en_rara} especies   |   Total: {total_general} especies"
    ]
})

# Texto superior
texto = alt.Chart(resumen).mark_text(
    align='left',
    fontSize=16
).encode(
    text='texto:N'
).properties(
    width=800,
    height=40
)

# Crear gráfico principal
grafico = alt.Chart(conteo).mark_bar().encode(

    # Regiones
    y=alt.Y(
        "REGIÓN:N",
        sort="-x",
        title="Región"
    ),

    # Cantidad de especies
    x=alt.X(
        "TOTAL:Q",
        title="Cantidad de especies"
    ),

    # Colores por categoría
    color=alt.Color(
        "CATEGORÍA VIGENTE:N",
        title="Categoría",
        scale=alt.Scale(
            domain=["CR", "EN", "EN-RARA"],
            range=[
                "#52b788",   # Verde → CR
                "#6a4c93",   # Morado → EN
                "	#0099FF"    # Azul → EN-RARA
            ]
        )
    ),

    # Tooltip interactivo
    tooltip=[
        alt.Tooltip("REGIÓN:N", title="Región"),
        alt.Tooltip("CATEGORÍA VIGENTE:N", title="Categoría"),
        alt.Tooltip("TOTAL:Q", title="Cantidad")
    ]

).properties(
    width=800,
    height=500,
    title="Especies amenazadas por región según categoría de conservación"
)

# Unir resumen + gráfico
final = alt.vconcat(
    texto,
    grafico
)

# Mostrar gráfico
final

# Guardar HTML
final.save("visualizacion.html")

# Guardar PNG
final.save("visualizacion.png")

# Convertir PNG a JPG
img = Image.open("visualizacion.png")
rgb = img.convert("RGB")
rgb.save("visualizacion.jpg")

final

alt.VConcatChart(...)